# Entrenamiento en Google Colab — configuración `v8s_800_colab`

Reproduce en Colab **la configuración que mejor precisión dio** en el registro
del proyecto (`models/models.csv`, rank 1):

| Parámetro | Valor | Por qué |
|---|---|---|
| Arquitectura | `yolov8s-seg.pt` | small supera a nano en +8.8 % de box mAP50; medium sobreajusta |
| Resolución | `800` | +3.2 % sobre 640 px |
| Oversampling | `3×` | repite en `train.txt` las imágenes de clases débiles |
| Épocas | `70` | `patience 20` |
| Batch | `8` | |
| Optimizador | `AdamW`, `lr0 0.002`, `cos_lr` | AdamW fijo: con `optimizer='auto'` Ultralytics ignora `lr0` |
| Semilla | `42` | mismo split que en local |
| copy-paste | `0.0` | en small no aportó nada |

Métricas de referencia sobre `test` (40 imágenes): **box mAP50 0.5674**,
**mask mAP50 0.5259**.

**Coste.** El entrenamiento original tardó **2 h 43 min** en Colab (70 épocas,
~140 s/época). Con GPU T4 cuenta con ese orden de magnitud; sin GPU no lo
intentes. Las celdas guardan un checkpoint en Drive cada pocas épocas, y hay
una celda de reanudación por si se corta la sesión.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de
ejecución → GPU`.

## Lo que necesitas en Drive

```
MyDrive/pucp-yolo/
└── dataset/
    ├── images/
    ├── labels/
    ├── data.yaml
    └── benchmark.txt
```

`benchmark.txt` importa: sus imágenes se excluyen del split. Si falta, el
reparto train/val/test no será el mismo que en local y las métricas dejarán de
ser comparables con las del registro.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi

## 2. Instalar dependencias

Se fija la misma versión de Ultralytics que usa el proyecto en local. Colab ya
trae torch con CUDA, así que no se toca.

In [ ]:
!pip install -q ultralytics==8.4.61

import torch
print(f'torch {torch.__version__}   CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('AVISO: sin GPU. Cambia el entorno de ejecución antes de continuar.')

## 3. Montar Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 4. Parámetros

Todo lo configurable está en esta celda. Los hiperparámetros son los del
modelo rank 1 (`models/history/v8s_800_colab/args.yaml`): cambiarlos deja de
reproducir esa configuración.

In [ ]:
# --- Drive ---
DRIVE_ROOT     = '/content/drive/MyDrive/pucp-yolo'
# Carpeta con images/, labels/, data.yaml y benchmark.txt.
DATASET_SOURCE = f'{DRIVE_ROOT}/dataset'
OUTPUT_DIR     = f'{DRIVE_ROOT}/resultados'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
CODE_ZIP       = ''                            # zip del repo, solo si git clone falla

# --- Código del proyecto ---
REPO_URL = 'https://github.com/Mike97179/pucp-project.git'
REPO_DIR = '/content/pucp-project'

# --- Entorno de trabajo: structures/config.py espera BASE_PATH = /content ---
BASE_PATH   = '/content'
DATASET_DIR = f'{BASE_PATH}/dataset'
RUNS_PATH   = f'{BASE_PATH}/runs_colab'

# Nombre de la corrida. El nombre definitivo del modelo (model_N) lo decide el
# registro al darlo de alta en local, no aquí.
RUN_NAME = 'v8s_800_over3x_colab'

# --- Configuración ganadora: v8s_800_colab, mask_mAP50 0.5259 ---
BASE_MODEL = 'yolov8s-seg.pt'
IMGSZ      = 800
BATCH      = 8
EPOCHS     = 70
PATIENCE   = 20
LR0        = 0.002
OPTIMIZER  = 'AdamW'
SEED       = 42
OVERSAMPLE = 3      # factor máximo de repetición de clases débiles
COPY_PASTE = 0.0    # en small no aportó nada

# Cada cuántas épocas se copia el checkpoint a Drive.
CHECKPOINT_EVERY = 5

print(f'Corrida    : {RUN_NAME}')
print(f'Modelo base: {BASE_MODEL}   imgsz: {IMGSZ}   batch: {BATCH}')
print(f'Épocas     : {EPOCHS}   lr0: {LR0}   optimizer: {OPTIMIZER}')

## 5. Traer el código del proyecto

Se clona el repositorio para usar exactamente los mismos módulos que en local
(`structures/`): el split, el oversampling y la evaluación son los del
proyecto, no una copia reescrita aquí.

Si el repositorio es privado y `git clone` falla, sube un zip del proyecto a
Drive y pon su ruta en `CODE_ZIP`.

In [ ]:
import os
import shutil
import subprocess
import zipfile

if os.path.isdir(os.path.join(REPO_DIR, 'structures')):
    print(f'El código ya está en {REPO_DIR}')
elif CODE_ZIP and os.path.isfile(CODE_ZIP):
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall('/content/_code_tmp')
    root = next(os.path.dirname(os.path.join(base, ''))
                for base, dirs, _ in os.walk('/content/_code_tmp')
                if 'structures' in dirs)
    shutil.move(root, REPO_DIR)
    print(f'Código descomprimido desde {CODE_ZIP}')
else:
    result = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
                            capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(
            f'No se pudo clonar {REPO_URL}:\n{result.stderr}\n'
            f'Si el repositorio es privado, sube un zip del proyecto a Drive '
            f'y ponlo en CODE_ZIP.')
    print(f'Repositorio clonado en {REPO_DIR}')

print(sorted(os.listdir(REPO_DIR)))

## 6. Preparar el dataset

El dataset no está en el repositorio (366 MB), así que sale de Drive. Se copia
a `/content/dataset`, que es donde `config.py` lo busca cuando `BASE_PATH` es
`/content`.

El dataset se copia al disco local de Colab en vez de entrenar leyendo desde
Drive, que sería mucho más lento. La copia de 400 imágenes desde Drive tarda
unos minutos; solo hace falta una vez por sesión.

Tres cosas que esta celda arregla y que si no dan problemas raros:

- `data.yaml` trae la ruta absoluta de la máquina local: se reescribe a
  `/content/dataset`.
- `labels.cache` de Ultralytics guarda rutas locales: se borra.
- si falta `benchmark.txt`, el split no coincidirá con el de local: se avisa.

In [ ]:
import glob

import yaml


def _dataset_root(path):
    """Carpeta que contiene images/ y labels/, esté o no anidada un nivel."""
    for base, dirs, _ in os.walk(path):
        if 'images' in dirs and 'labels' in dirs:
            return base
    raise SystemExit(f'En {path} no hay ninguna carpeta con images/ y labels/.')


if os.path.isdir(os.path.join(DATASET_DIR, 'images')):
    print(f'El dataset ya está en {DATASET_DIR}')
else:
    if not os.path.isdir(DATASET_SOURCE):
        raise SystemExit(f'No existe la carpeta {DATASET_SOURCE}. '
                         f'Revisa la ruta en Drive.')

    source = _dataset_root(DATASET_SOURCE)
    print(f'Copiando {source} -> {DATASET_DIR} (tarda unos minutos)...')
    shutil.copytree(source, DATASET_DIR, dirs_exist_ok=True)
    print('Dataset copiado.')

# El cache de Ultralytics apunta a las rutas de la máquina donde se creó.
for cache in glob.glob(os.path.join(DATASET_DIR, '*.cache')):
    os.remove(cache)
    print(f'Cache borrado: {os.path.basename(cache)}')

# data.yaml: si la carpeta no lo trae, se usa el del repositorio (sí está
# versionado).
yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
if not os.path.isfile(yaml_path):
    shutil.copy2(os.path.join(REPO_DIR, 'dataset', 'data.yaml'), yaml_path)

with open(yaml_path) as f:
    spec = yaml.safe_load(f)

spec['path'] = DATASET_DIR
spec.setdefault('train', 'train.txt')
spec.setdefault('val',   'val.txt')
spec.setdefault('test',  'test.txt')

with open(yaml_path, 'w') as f:
    yaml.safe_dump(spec, f, sort_keys=False, allow_unicode=True)

n_images = len(glob.glob(os.path.join(DATASET_DIR, 'images', '*')))
n_labels = len(glob.glob(os.path.join(DATASET_DIR, 'labels', '*.txt')))
print(f'\nImágenes : {n_images}')
print(f'Labels   : {n_labels}')
print(f'Clases   : {spec["nc"]} -> {spec["names"]}')

if not os.path.isfile(os.path.join(DATASET_DIR, 'benchmark.txt')):
    print('\nAVISO: falta benchmark.txt. Sus imágenes se excluyen del split, '
          'así que sin él\n       el reparto train/val/test no será el mismo '
          'que en local y las métricas\n       no serán comparables con las '
          'del registro.')

## 7. Cargar el software del proyecto

`config.py` decide sus rutas al importarse, y los módulos las capturan como
valores por defecto. Por eso el directorio de trabajo se fija **antes** del
`import`: a partir de aquí `config.BASE_PATH` es `/content`.

In [ ]:
import sys

os.chdir(BASE_PATH)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from structures import config, data, evaluate, models, plots, train
from structures.commands import validate

if config.BASE_PATH != BASE_PATH:
    raise SystemExit(f'config.BASE_PATH es {config.BASE_PATH} y se esperaba '
                     f'{BASE_PATH}: reinicia el entorno y ejecuta esta celda '
                     f'antes de importar nada de structures.')

print(f'dataset : {config.DATASET_PATH}')
print(f'data.yaml: {config.YAML_PATH}')
print(f'modelos : {config.MODELS_DIR}')

## 8. Validar el dataset

Ultralytics se salta en silencio las anotaciones que no entiende y entrena
igual; el síntoma aparece horas después como una clase que no aprende. Esta
celda cruza imágenes y labels y revisa cada polígono.

In [ ]:
problems = validate.run(None)
if problems:
    print('\nHay problemas en el dataset: revísalos antes de entrenar.')

## 9. Split y oversampling

El split se regenera desde la semilla 42, igual que en local, así que el
reparto es el mismo con el mismo dataset. El oversampling solo repite líneas
en `train.txt` —no duplica archivos— y `val`/`test` se quedan intactos.

In [ ]:
class_names = config.load_class_names()

print('Generando split train/val/test (semilla fija)...')
train_imgs, val_imgs, test_imgs = data.generate_split(seed=SEED)

if OVERSAMPLE > 1:
    print(f'\nAplicando oversampling (factor máx {OVERSAMPLE}x)...')
    train_over = data.apply_oversampling(train_imgs, class_names,
                                         max_factor=OVERSAMPLE)
    data.write_split(train_over, val_imgs, test_imgs)

## 10. Checkpoints en Drive

Casi tres horas de entrenamiento sobrevivirán mejor si cada pocas épocas los
pesos acaban en Drive. El callback se registra en Ultralytics y copia
`last.pt`, `best.pt`, `args.yaml` y `results.csv` a
`CHECKPOINT_DIR/RUN_NAME/`.

In [ ]:
from ultralytics.utils import callbacks

CHECKPOINT_FILES = ['weights/last.pt', 'weights/best.pt',
                    'args.yaml', 'results.csv']


def sync_checkpoint(trainer):
    """Copia el estado del entrenamiento a Drive cada CHECKPOINT_EVERY épocas."""
    epoch = trainer.epoch + 1
    if epoch % CHECKPOINT_EVERY and epoch != trainer.epochs:
        return

    target = os.path.join(CHECKPOINT_DIR, RUN_NAME)
    os.makedirs(os.path.join(target, 'weights'), exist_ok=True)

    copied = 0
    for name in CHECKPOINT_FILES:
        source = os.path.join(trainer.save_dir, name)
        if os.path.isfile(source):
            shutil.copy2(source, os.path.join(target, name))
            copied += 1

    print(f'  [checkpoint] época {epoch}: {copied} archivos en {target}')


hooks = callbacks.default_callbacks['on_fit_epoch_end']
if not any(h.__name__ == 'sync_checkpoint' for h in hooks):
    hooks.append(sync_checkpoint)

os.makedirs(os.path.join(CHECKPOINT_DIR, RUN_NAME), exist_ok=True)
print(f'Checkpoints cada {CHECKPOINT_EVERY} épocas en '
      f'{os.path.join(CHECKPOINT_DIR, RUN_NAME)}')

## 11. Entrenar

Aquí van las 2-3 horas. `train.train()` es el mismo wrapper que usa el comando
`train` en local, con los hiperparámetros de la configuración ganadora.

Pase lo que pase, `train.txt` se deja sin repeticiones al terminar: si se
quedara con el oversampling aplicado, la siguiente corrida lo usaría sin
enterarse.

Si da error de memoria, bajar `BATCH` a 4 es lo primero que funciona — pero
entonces ya no es la misma configuración y las métricas no son comparables con
las del registro.

In [ ]:
extra = {'copy_paste': COPY_PASTE} if COPY_PASTE else {}

train.check_environment()

try:
    weights = train.train(
        run_name   = RUN_NAME,
        runs_path  = RUNS_PATH,
        base_model = BASE_MODEL,
        epochs     = EPOCHS,
        imgsz      = IMGSZ,
        batch      = BATCH,
        lr0        = LR0,
        patience   = PATIENCE,
        optimizer  = OPTIMIZER,
        seed       = SEED,
        exist_ok   = False,
        **extra
    )
finally:
    print('\nRestaurando train.txt al split original...')
    data.write_split(train_imgs, val_imgs, test_imgs)

print(f'\nPesos: {weights}')

### Si se cortó la sesión

Vuelve a ejecutar las celdas 1 a 10 (instalación, dataset, split) y ejecuta
**esta** celda en lugar de la de entrenamiento: recupera el último checkpoint
de Drive y sigue desde la época en la que se quedó.

In [ ]:
from ultralytics import YOLO

checkpoint = os.path.join(CHECKPOINT_DIR, RUN_NAME)
run_dir    = os.path.join(RUNS_PATH, RUN_NAME)
last       = os.path.join(run_dir, 'weights', 'last.pt')

if not os.path.isfile(os.path.join(checkpoint, 'weights', 'last.pt')):
    raise SystemExit(f'No hay checkpoint en {checkpoint}: no hay nada que '
                     f'reanudar, entrena desde cero.')

shutil.copytree(checkpoint, run_dir, dirs_exist_ok=True)
print(f'Checkpoint restaurado en {run_dir}')

try:
    YOLO(last).train(resume=True)
finally:
    print('\nRestaurando train.txt al split original...')
    data.write_split(train_imgs, val_imgs, test_imgs)

weights = os.path.join(run_dir, 'weights', 'best.pt')
print(f'\nPesos: {weights}')

## 12. Evaluar sobre `test`

Las métricas salen de `model.val()` sobre el split de test, nunca de la última
fila de `results.csv` —esa es la última época, no la mejor—. Son las cifras
que después van al registro.

In [ ]:
run_dir = os.path.join(RUNS_PATH, RUN_NAME)

metrics = evaluate.validate(weights, split='test', verbose=True)
totals  = evaluate.summary(metrics)

print('\n' + '=' * 70)
print(' MÉTRICAS POR CLASE (segmentación)')
print('=' * 70)
df_classes = evaluate.per_class_metrics(metrics, class_names)
print(df_classes.to_string(index=False))

df_classes.to_csv(os.path.join(run_dir, 'per_class_metrics.csv'), index=False)

plots.map_per_class(
    df_classes,
    os.path.join(run_dir, 'per_class_metrics.png'),
    title=f'mAP50 por clase — {RUN_NAME}')

plots.confusion_matrix(
    evaluate.confusion_matrix(metrics),
    config.matrix_labels(class_names),
    os.path.join(run_dir, 'confusion_matrix_test.png'),
    title=f'Matriz de confusión — {RUN_NAME}')

print('\n' + '=' * 70)
print(f' RESUMEN — {RUN_NAME}')
print('=' * 70)
print(f'  mAP50 detección    : {totals["mAP50_detection"]:.4f}   '
      f'(referencia v8s_800_colab: 0.5674)')
print(f'  mAP50 segmentación : {totals["mAP50_segmentation"]:.4f}   '
      f'(referencia v8s_800_colab: 0.5259)')
print(f'  mAP50-95 (seg)     : {totals["mAP5095_segmentation"]:.4f}')
print('=' * 70)

## 13. Guardar los resultados en Drive

Se copia lo que hace falta para dar el modelo de alta en local: los pesos y los
mismos archivos de historial que guarda el comando `train`
(`models.HISTORY_FILES`).

In [ ]:
target = os.path.join(OUTPUT_DIR, RUN_NAME)
os.makedirs(target, exist_ok=True)

shutil.copy2(os.path.join(run_dir, 'weights', 'best.pt'),
             os.path.join(target, f'{RUN_NAME}.pt'))

extra_files = ['per_class_metrics.csv', 'per_class_metrics.png',
               'confusion_matrix_test.png']

copied = ['best.pt']
for name in models.HISTORY_FILES + extra_files:
    source = os.path.join(run_dir, name)
    if os.path.isfile(source):
        shutil.copy2(source, os.path.join(target, name))
        copied.append(name)

snippet = f'''from structures import models

models.register_new(
    models.next_model_name(),
    'RUTA/AL/{RUN_NAME}.pt',
    run_dir      = 'RUTA/A/LA/CARPETA/DESCARGADA',
    architecture = '{os.path.splitext(BASE_MODEL)[0]}',
    imgsz        = {IMGSZ},
    data         = 'oversample_{OVERSAMPLE}x',
    box_mAP50    = {totals["mAP50_detection"]},
    mask_mAP50   = {totals["mAP50_segmentation"]},
    origin       = '{RUN_NAME}',
    notes        = 'Entrenado en Colab',
)'''

with open(os.path.join(target, 'registrar.txt'), 'w') as f:
    f.write(snippet + '\n')

print(f'Guardado en {target}:')
for name in copied:
    print(f'  {name}')

print('\nPara darlo de alta en local (después de descargar la carpeta):\n')
print(snippet)

## 14. Dar el modelo de alta en local

Descarga de Drive la carpeta `resultados/v8s_800_over3x_colab/` y, en el
proyecto local con el entorno activado:

```bash
source .pucp-project/bin/activate
python - <<'PY'
from structures import models
# ...pega aquí el snippet que imprimió la celda 13 (también está en
# registrar.txt), con las rutas de la carpeta descargada
PY
```

`register_new()` copia los pesos a `models/<nombre>.pt`, guarda el historial en
`models/history/<nombre>/`, añade la fila al CSV y aplica el leaderboard de 5:
si el modelo nuevo entra, el peor se archiva en `models/archive/`. El `rank` se
recalcula solo desde `mask_mAP50`, no hay que tocarlo a mano.

Después:

```bash
python pucp_segmentation.py ranking      # ¿cambió el rank 1?
python pucp_segmentation.py benchmark    # comparación visual con los demás
```

Si el modelo nuevo se queda con el rank 1, hay que actualizar el README, el
bloque destacado de `comparativa_modelos.html` y la columna `notes` del CSV, y
recalcular las cifras del HTML desde su matriz de confusión.